In [3]:
import torch
from torch import nn
from torchvision import models
from torchvision.models import MobileNet_V3_Large_Weights

In [ ]:
class LeafClassifier(nn.Module):
    def __init__(self, num_classes, DROPOUT_RATE=0.25):
        super(LeafClassifier, self).__init__()
        # Use the latest weights for better initialization
        self.model = models.mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.DEFAULT)
        
        # Freeze fewer layers - only freeze the first 50% of layers
        # This allows for better fine-tuning to your specific leaf dataset
        total_params = len(list(self.model.parameters()))
        for param in list(self.model.parameters())[:total_params//2]:
            param.requires_grad = False
            
        # Replace with a simpler, more robust classifier
        in_features = self.model.classifier[0].in_features
        self.model.classifier = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.BatchNorm1d(512),  # Add batch normalization for stability
            nn.ReLU(inplace=True),
            nn.Dropout(DROPOUT_RATE),
            
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(DROPOUT_RATE),
            
            nn.Linear(256, num_classes)
        )
        # Apply better weight initialization
        self._initialize_weights()

    def forward(self, x):
        return self.model(x)
    
    def _initialize_weights(self):
        """Apply improved weight initialization to newly added layers"""
        for m in self.model.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

In [7]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


# Load your saved model (make sure to define LeafClassifier first)
num_classes = 10
model = LeafClassifier(num_classes, DROPOUT_RATE=0.25).to(DEVICE)  # Replace your_num_classes
model.load_state_dict(torch.load('mobilenetv3_best_weights7.pth'))  # Replace with your .pth file path
model.eval()

C:\Users\Asus TUF -PC\AppData\Local\Temp\ipykernel_44948\2754206433.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('mobilenetv3_best_we

LeafClassifier(
  (model): MobileNetV3(
    (features): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
        (2): Hardswish()
      )
      (1): InvertedResidual(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=16, bias=False)
            (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
            (2): ReLU(inplace=True)
          )
          (1): Conv2dNormActivation(
            (0): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
            (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
          )
        )
      )
      (2): InvertedResidual(
        (block): Sequential(
          (0): Conv2dNormActivati

In [10]:
dummy_input = torch.randn(1, 3, 224, 224).to(DEVICE)  # Add .to(DEVICE)


In [11]:
# Export to ONNX
torch.onnx.export(
    model,
    dummy_input,
    'leaf_classifier_81.onnx',
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}},
    opset_version=12
)

print("Model successfully converted to ONNX format: leaf_classifier.onnx")

Model successfully converted to ONNX format: leaf_classifier.onnx
